In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed"

destination_master = pd.read_csv(
    DATA_PATH / "destination_master.csv"
)

print("Destination dataset shape:", destination_master.shape)

display(
    destination_master[
        destination_master["Location_Name"].str.contains(
            "Bundala|Udawalawe|Sinharaja|Wilpattu",
            case=False,
            na=False
        )
    ][
        [
            "Location_Name",
            "Located_City",
            "Destination_District",
            "Mapped_District",
            "Geography_Notes"
        ]
    ]
)

Destination dataset shape: (76, 12)


,Location_Name,Located_City,Destination_District,Mapped_District,Geography_Notes
8,Bundala National Park,Weligatta,NaN,Hambantota,NaN
66,Sinharaja Forest Reserve,Deniyaya,NaN,NaN,Spans multiple districts; exact coverage needs...
71,Udawalawe National Park,Embilipitiya,Ratnapura / Moneragala,NaN,Spans Ratnapura and Moneragala districts
74,Wilpattu National Park,Saliyapura,NaN,NaN,Spans multiple districts; exact coverage needs...


In [2]:
# Check destination district mappings

mapping_summary = destination_master[
    [
        "Location_Name",
        "Located_City",
        "Destination_District",
        "Mapped_District"
    ]
].copy()

mapping_summary["Mapping_Status"] = mapping_summary[
    "Mapped_District"
].apply(
    lambda x: "Mapped" if pd.notna(x) and str(x).strip() else "Missing"
)

print("Mapping status:")
print(mapping_summary["Mapping_Status"].value_counts())

print("\nDestinations requiring mapping review:")

display(
    mapping_summary[
        mapping_summary["Mapping_Status"] == "Missing"
    ]
)

Mapping status:
Mapping_Status
Mapped     73
Missing     3
Name: count, dtype: int64

Destinations requiring mapping review:


,Location_Name,Located_City,Destination_District,Mapped_District,Mapping_Status
66,Sinharaja Forest Reserve,Deniyaya,NaN,NaN,Missing
71,Udawalawe National Park,Embilipitiya,Ratnapura / Moneragala,NaN,Missing
74,Wilpattu National Park,Saliyapura,NaN,NaN,Missing


In [3]:
# Keep uncertain district mappings missing
uncertain_destinations = [
    "Sinharaja Forest Reserve",
    "Udawalawe National Park",
    "Wilpattu National Park"
]

destination_master.loc[
    destination_master["Location_Name"].isin(uncertain_destinations),
    "Mapped_District"
] = pd.NA

# Check the result
display(
    destination_master[
        destination_master["Location_Name"].isin(uncertain_destinations)
    ][
        [
            "Location_Name",
            "Located_City",
            "Destination_District",
            "Mapped_District"
        ]
    ]
)

,Location_Name,Located_City,Destination_District,Mapped_District
66,Sinharaja Forest Reserve,Deniyaya,NaN,<NA>
71,Udawalawe National Park,Embilipitiya,Ratnapura / Moneragala,<NA>
74,Wilpattu National Park,Saliyapura,NaN,<NA>


In [4]:
# Save the updated destination master dataset
output_path = DATA_PATH / "destination_master.csv"

destination_master.to_csv(output_path, index=False)

print("Updated destination_master.csv saved successfully!")
print("Dataset shape:", destination_master.shape)

# Verify missing district mappings
missing = destination_master[
    destination_master["Mapped_District"].isna()
]

print("Destinations with missing district mappings:", len(missing))

display(
    missing[
        ["Location_Name", "Located_City", "Mapped_District"]
    ]
)

Updated destination_master.csv saved successfully!
Dataset shape: (76, 12)
Destinations with missing district mappings: 3


,Location_Name,Located_City,Mapped_District
66,Sinharaja Forest Reserve,Deniyaya,<NA>
71,Udawalawe National Park,Embilipitiya,<NA>
74,Wilpattu National Park,Saliyapura,<NA>


In [5]:
display(
    destination_master[
        destination_master["Location_Name"].str.contains(
            "Bentota Beach", case=False, na=False
        )
    ][
        [
            "Location_Name",
            "Located_City",
            "Destination_District",
            "Mapped_District",
            "Geography_Notes"
        ]
    ]
)

,Location_Name,Located_City,Destination_District,Mapped_District,Geography_Notes
5,Bentota Beach,Bentota,Galle,Galle,NaN


In [6]:
import pandas as pd

hotel_master = pd.read_csv(DATA_PATH / "hotel_master.csv")

bentota_hotels = hotel_master[
    hotel_master["Address"].fillna("").str.contains(
        "Bentota", case=False, na=False
    )
    | hotel_master["Name"].fillna("").str.contains(
        "Bentota", case=False, na=False
    )
]

display(
    bentota_hotels[
        ["Name", "Type", "District", "Address"]
    ].head(15)
)

,Name,Type,District,Address
8,CLUB VILLA,Boutique Hotels,Galle,"NO.138/15, GALLE ROAD, BENTOTA"
28,ROCK VILLA,Boutique Villas,Galle,"NEELISGEWATTA,ALAWATHUGODA,BENTOTA."
42,WATERSIDE BENTOTA,Boutique Villas,Galle,"YATHRAMULLA,BENTOTA"
63,PARADISE ROAD THE VILLA BENTOTA,Boutique Villas,Galle,"138/18, GALLE ROAD,BENTOTA"
96,RIVER SIDE INN FUJI,Bangalows,Galle,"81,CIRCULAR ROAD,OBADAWATTA,BENTOTA"
98,WESTROM VILLA,Bangalows,Galle,"ROBOLGODA, BENTOTA."
118,CINNAMON VILLA,Bangalows,Galle,"KADJUGASDUWA ESTATE,MIRISWATTA, HABURUGALA, BE..."
132,RIVERSIDE BUNGALOW,Bangalows,Galle,"RIVER SIDE BUNGLOW, NO 55, PARADISE ISLAND ROA..."
233,SUPUN VILLA,Bangalows,Galle,"SUPUN VILLA, PITARAMA, BENTOTA"
251,SERENDIVA,Bangalows,Galle,"39,RANTHOTUWILA ROAD,HABURUGALA,BENTOTA."


In [7]:

print("Accommodation records by district:")
print(hotel_master["District"].value_counts(dropna=False).head(15))

print("\nBentota-related accommodation records by district:")
print(
    bentota_hotels["District"]
    .value_counts(dropna=False)
)


Accommodation records by district:
District
Colombo         297
Galle           284
Kandy           266
Gampaha         207
Kalutara        132
Badulla         116
Nuwara Eliya     93
Matara           88
Matale           86
Hambantota       76
Ampara           62
Anuradhapura     58
Puttalam         54
Kurunegala       48
Batticaloa       44
Name: count, dtype: int64

Bentota-related accommodation records by district:
District
Galle       34
Kalutara     1
Name: count, dtype: int64


In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("App folder exists:", (PROJECT_ROOT / "app").exists())
print("Hotel recommender exists:", (PROJECT_ROOT / "app" / "hotel_recommender.py").exists())

Project root: c:\Users\chami\OneDrive\Desktop\Tourist_Recommendation_System
App folder exists: True
Hotel recommender exists: True


In [10]:
from app.hotel_recommender import recommend_hotels

results = recommend_hotels(
    destination_name="Bentota Beach",
    top_n=10
)

print("Number of accommodation results:", len(results))
display(results)

Number of accommodation results: 10


,Name,Type,District,Grade,Rooms,Address
0,BENTOTA BEACH HOTEL,Classified Hotels( 1-5 Star),Galle,FOUR,133,"NATIONAL HOLIDAY RESORT, BENTOTA"
1,CLUB VILLA,Boutique Hotels,Galle,UNKNOWN,15,"NO.138/15, GALLE ROAD, BENTOTA"
2,HOTEL SUSANTHAS,Classified Hotels( 1-5 Star),Galle,ONE,18,"NATIONAL HOLIDAY RESORT RD,PITARAMBA,BENTOTA"
3,HOTEL PANCHI VILLA,Guest Houses,Galle,A,8,"KURUWATTA, WARAHENA, BENTOTA."
4,SERENDIVA,Bangalows,Galle,DELUXE,4,"39,RANTHOTUWILA ROAD,HABURUGALA,BENTOTA."
5,MARINA BENTOTA,Guest Houses,Galle,A,13,"PARADISE ISLAND, NATIONAL HOLIDAY RESORT, BENT..."
6,TARU VILLAS-RIVER COTTAGE,Bangalows,Galle,DELUXE,2,"111,BENTOTA CIRCULAR ROAD, BENTOTA 80500"
7,AVANI BENTOTA RESORT & SPA,Classified Hotels( 1-5 Star),Galle,TWO,90,"NATIONAL HOLIDAY RESORT,BENTOTA"
8,ROCK VILLA,Boutique Villas,Galle,UNKNOWN,7,"NEELISGEWATTA,ALAWATHUGODA,BENTOTA."
9,WATERSIDE BENTOTA,Boutique Villas,Galle,UNKNOWN,6,"YATHRAMULLA,BENTOTA"


In [11]:

# Check how many returned accommodations mention Bentota
address_match = results["Address"].fillna("").str.contains(
    "Bentota", case=False, na=False
)

name_match = results["Name"].fillna("").str.contains(
    "Bentota", case=False, na=False
)

print("Total recommendations:", len(results))
print("Addresses mentioning Bentota:", int(address_match.sum()))
print("Names mentioning Bentota:", int(name_match.sum()))
print("Records mentioning Bentota in name or address:",
      int((address_match | name_match).sum()))

display(
    results.assign(
        Bentota_Match=address_match | name_match
    )[["Name", "District", "Address", "Bentota_Match"]]
)


Total recommendations: 10
Addresses mentioning Bentota: 10
Names mentioning Bentota: 4
Records mentioning Bentota in name or address: 10


,Name,District,Address,Bentota_Match
0,BENTOTA BEACH HOTEL,Galle,"NATIONAL HOLIDAY RESORT, BENTOTA",True
1,CLUB VILLA,Galle,"NO.138/15, GALLE ROAD, BENTOTA",True
2,HOTEL SUSANTHAS,Galle,"NATIONAL HOLIDAY RESORT RD,PITARAMBA,BENTOTA",True
3,HOTEL PANCHI VILLA,Galle,"KURUWATTA, WARAHENA, BENTOTA.",True
4,SERENDIVA,Galle,"39,RANTHOTUWILA ROAD,HABURUGALA,BENTOTA.",True
5,MARINA BENTOTA,Galle,"PARADISE ISLAND, NATIONAL HOLIDAY RESORT, BENT...",True
6,TARU VILLAS-RIVER COTTAGE,Galle,"111,BENTOTA CIRCULAR ROAD, BENTOTA 80500",True
7,AVANI BENTOTA RESORT & SPA,Galle,"NATIONAL HOLIDAY RESORT,BENTOTA",True
8,ROCK VILLA,Galle,"NEELISGEWATTA,ALAWATHUGODA,BENTOTA.",True
9,WATERSIDE BENTOTA,Galle,"YATHRAMULLA,BENTOTA",True
